# 03 — Offline sandbox smoke (mock)

Shows that the prepared corpus can drive the sandbox eval machinery **without a model server**.

1. Load and validate `prep_manifest.json`, preparing it first if notebook 02 has not run.
2. Show a **dry-run plan**: what would run, its row count and dataset fingerprint.
3. Run the **mock sorter** eval on the prepared fixtures and on the class-balanced slice.
4. Run a **mock pipeline pilot** through the vendored LangGraph pipeline.
5. Confirm that the eval records went to the **notebook experiment log**, not `reports/`.

**What mock means:** the mock sorter echoes each row's gold class, and the mock pipeline swaps in a fake LLM client. A score of 1.0 therefore checks the **plumbing**: loading, graph execution, scoring and logging. It says **nothing about model quality**. A real local model needs `sandbox pilot --local` against a running Ollama.

**Cost:** none. Writes go to `data/` (run store, archive, SQLite) and `data/runtime/notebooks/experiment_log.jsonl`. Both are gitignored.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Works with or without `pip install -e .`: fall back to the checkout's src/ when needed.
try:
    import mailroom_sandbox  # noqa: F401
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "src" / "mailroom_sandbox").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import mailroom_sandbox  # noqa: F401

from mailroom_sandbox.notebook import bootstrap

ctx = bootstrap(mock=True)
print(json.dumps(ctx.as_dict(), indent=2))

{
  "root": "/demo/checkout",
  "profile": "ollama",
  "mock": true,
  "observability": "none",
  "data_dir": "/demo/runtime/data",
  "prepared_dir": "/demo/runtime/data/runtime/prepared",
  "experiment_log": "/tmp/notebook-demos-03gnk3ib/log/experiment_log.jsonl"
}


In [2]:
try:
    import pandas as pd
except ImportError as exc:  # pragma: no cover - guidance for a bare [dev] install
    raise ImportError('pandas is missing: pip install -e ".[dev,notebooks]"') from exc

pd.set_option("display.max_colwidth", 80)
print(f"pandas {pd.__version__}")


pandas 3.1.1


## 1. Load and validate the prepared corpus

In [3]:
from mailroom_sandbox.datasets import load_jsonl
from mailroom_sandbox.notebook import check_prepared_artifacts
from mailroom_sandbox.prep import prepare_offline_datasets

manifest_path = ctx.prepared_dir / "prep_manifest.json"
if not manifest_path.is_file():
    print("prepared corpus missing; running prepare_offline_datasets() (notebook 02 does this step by step)")
    prepare_offline_datasets()

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
problems = check_prepared_artifacts(manifest, ctx.root)
assert not problems, "prepared corpus is stale or broken; rerun notebook 02:\n" + "\n".join(problems)

fixtures = load_jsonl(ctx.root / manifest["artifacts"]["fixtures"])
balanced = load_jsonl(ctx.root / manifest["artifacts"]["sorter_pilot_balanced"])
print(json.dumps({"prepared_at": manifest["prepared_at"], "fingerprint": manifest["fingerprint"],
                  "counts": manifest["counts"]}, indent=2))

{
  "prepared_at": "2026-11-20T12:00:20.134667+00:00",
  "fingerprint": "418eb49b0465",
  "counts": {
    "fixtures": 11,
    "hf": 7,
    "agents": {
      "judge": 2,
      "arbiter": 2,
      "boss": 2,
      "intake": 3,
      "pdf_transcriber": 1,
      "image_extractor": 1
    },
    "sorter_pilot_balanced": 6
  }
}


## 2. Dry-run plan

With `dry_run=True` the runner returns its plan without activating a profile or logging anything. This is the cheap way to check row counts and the dataset fingerprint before a real run.

In [4]:
from mailroom_sandbox.eval.runners import run_pipeline_eval, run_sorter_eval

plans = [
    run_sorter_eval(mock=True, dry_run=True, rows=fixtures, profile=ctx.profile),
    run_pipeline_eval(mock=True, dry_run=True, rows=fixtures, sample=4),
]
pd.DataFrame(plans)

,task,n,mock,prompt_version,profile,model,fingerprint,connected
0,sorter,11,True,NaN,ollama,NaN,ac483bde2aab,NaN
1,pipeline,4,True,NaN,NaN,NaN,e506bd4057dc,True


## 3. Mock sorter eval

Both runs run inside `experiment_log_redirect()`, so their records go to the notebook log. To log to `reports/experiment_log.jsonl` instead, start Jupyter with `SANDBOX_NOTEBOOK_LOG=reports`. Mock records there are flagged `mock: true`, but they still sit beside real runs.

In [5]:
from mailroom_sandbox.notebook import experiment_log_redirect

with experiment_log_redirect(ctx.experiment_log) as log_path:
    sorter_full = run_sorter_eval(
        mock=True, rows=fixtures, profile=ctx.profile, experiment_name="notebook_sorter_fixtures"
    )
    sorter_balanced = run_sorter_eval(
        mock=True, rows=balanced, profile=ctx.profile, experiment_name="notebook_sorter_balanced"
    )

METRICS = ["n", "exact_match", "f1_macro", "precision_macro", "recall_macro"]
sorter_table = pd.DataFrame(
    {
        name: {"n": result["n"], **{m: result["scores"].get(m) for m in METRICS[1:]}}
        for name, result in (("fixtures", sorter_full), ("balanced", sorter_balanced))
    }
).T
display(sorter_table)

assert sorter_full["n"] == len(fixtures) and sorter_balanced["n"] == len(balanced)
assert sorter_full["scores"]["exact_match"] == 1.0, "mock sorter echoes gold; anything else is a plumbing bug"
assert len({r["expected_doc_class"] for r in balanced}) == len(balanced), "balanced slice must be one row per class"

,n,exact_match,f1_macro,precision_macro,recall_macro
fixtures,11.0,1.0,1.0,1.0,1.0
balanced,6.0,1.0,1.0,1.0,1.0


In [6]:
per_class = pd.DataFrame(sorter_full["scores"].get("per_class") or {}).T
per_class if not per_class.empty else "no per-class breakdown in this scorer version"

'no per-class breakdown in this scorer version'

## 4. Mock pipeline pilot

Runs the first `SAMPLE` fixture rows through the full vendored graph: intake, classification, extraction, review routing, procedural report and archive. This is what `sandbox pilot --mock` does. Increase `SAMPLE` to cover more classes. The mock pipeline runs one document at a time.

Routing stays **blind** here: no ground truth reaches the graph. Only the mock LLM answers from each row's gold, so `routing_accuracy` and the automation numbers check plumbing and do not carry over to production.

The compliance fixtures are expected to stop at `review` with `unknown_doc_type: 'compliance_filing'`. The compliance specialist is retired from the vendored pipeline, so those rows also pull `extraction_overall` and `schema_valid_rate` below 1.0. A `langfuse_unavailable` warning only means the `[observability]` extra is not installed, and tracing is off here anyway.

In [7]:
from mailroom_sandbox.notebook import quiet_pipeline_logs

SAMPLE = min(4, len(fixtures))

# Keep the demo cell readable: expected compliance-fixture and relations warnings go to a buffer.
with quiet_pipeline_logs() as _log_buf, experiment_log_redirect(ctx.experiment_log):
    pipeline = run_pipeline_eval(
        mock=True,
        rows=fixtures,
        sample=SAMPLE,
        experiment_name="notebook_offline_pipeline",
        profile=ctx.profile,
    )
_noise = _log_buf.getvalue()
if "compliance_filing" in _noise or "unknown_doc_type" in _noise:
    print("pipeline warnings for retired compliance fixtures were suppressed (see markdown above)")

keys = ["n", "exact_match", "f1_macro", "stage_correct", "extraction_overall", "routing_accuracy",
        "schema_valid_rate", "parse_error_rate", "routed_to_human_review_rate"]
display(pd.Series({k: pipeline["scores"].get(k) for k in keys}, name="mock pipeline").to_frame())

docs = pd.DataFrame(pipeline.get("docs") or [])
cols = [c for c in ("id", "doc_type", "stage", "error") if c in docs.columns]
display(docs[cols] if cols else docs)

assert pipeline["n"] == SAMPLE
assert pipeline["scores"].get("parse_error_count", 0) == 0, "mock pipeline produced parse errors"


pipeline warnings for retired compliance fixtures were suppressed (see markdown above)


,mock pipeline
n,4.0
exact_match,1.0
f1_macro,1.0
stage_correct,1.0
extraction_overall,0.5
routing_accuracy,1.0
schema_valid_rate,0.5
parse_error_rate,0.0
routed_to_human_review_rate,0.5


,id,doc_type,stage,error
0,contract_msa,contract,archived,None
1,contract_nda,contract,archived,None
2,compliance_10k,compliance_filing,review,None
3,compliance_state,compliance_filing,review,None


## 5. Where the records went

In [8]:
from mailroom_sandbox.eval import experiment_log

records = load_jsonl(ctx.experiment_log)
mine = [r for r in records if str(r.get("experiment_name", "")).startswith("notebook_")]
print(f"{len(mine)} notebook records in {ctx.as_dict()['experiment_log']} ({len(records)} total)")
print("reports log untouched by this notebook:", experiment_log.jsonl_path() != ctx.experiment_log)
pd.DataFrame(mine)[["timestamp", "experiment_name", "task", "profile", "mock", "n", "dataset_fingerprint"]].tail(6)

3 notebook records in /tmp/notebook-demos-03gnk3ib/log/experiment_log.jsonl (3 total)
reports log untouched by this notebook: True


,timestamp,experiment_name,task,profile,mock,n,dataset_fingerprint
0,2026-11-20T12:00:22.581478+00:00,notebook_sorter_fixtures,sorter,ollama,True,11,ac483bde2aab
1,2026-11-20T12:00:22.663008+00:00,notebook_sorter_balanced,sorter,ollama,True,6,8671bb8d5133
2,2026-11-20T12:00:23.702139+00:00,notebook_offline_pipeline,pipeline,ollama,True,4,e506bd4057dc


## Next steps

The offline path works. Use [`04_experiment_log_explorer.ipynb`](04_experiment_log_explorer.ipynb) to browse the records.

To try a **real local model** (free, but it needs a running Ollama), run on the host:

```bash
sandbox up --compose-profile ollama
sandbox pull-models --profile ollama
sandbox health --profile ollama
sandbox pilot --local --profile ollama
```

Paid or remote work, such as Modal GPUs or OpenRouter, **never** starts from these notebooks. It goes through the spend gate: `sandbox run preflight --config <file>` first, then `sandbox run start` with a human's initials, after reading `CLAUDE.md`'s spend section.